# Charge storage and switching

Implicit finite differences resolve stored-charge removal after injection is switched off.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "figures" / "src").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate lecture-notes/figures/src")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "figures" / "src"))
from scipy.sparse import diags, eye
from scipy.sparse.linalg import spsolve
from scipy.constants import elementary_charge as q

length,D,tau=120e-6,1e-3,2e-6
x=np.linspace(0,length,181); dx=x[1]-x[0]; dt=0.1e-6
L=np.sqrt(D*tau)
u=np.sinh((length-x)/L)/np.sinh(length/L)*1e20
m=len(x)-2
Lap=diags((np.ones(m-1),-2*np.ones(m),np.ones(m-1)),(-1,0,1))/dx**2
M=(eye(m)-dt*(D*Lap-eye(m)/tau)).tocsr()
u_new=np.zeros_like(u); u_new[1:-1]=spsolve(M,u[1:-1])
print(f"one-step minimum density = {u_new.min():.3e} m^-3")

one-step minimum density = 0.000e+00 m^-3


In [2]:
from semphy.junctions import transient_minority_diffusion_fd

times=np.linspace(0,12e-6,121)
result=transient_minority_diffusion_fd(length,D,tau,1e20,times,points=181)
Qstored=result["stored_charge_c_m2"]
print(f"remaining charge fraction = {Qstored[-1]/Qstored[0]:.3e}")
assert np.min(result["excess_m3"]) >= 0
assert np.all(np.diff(Qstored) <= 1e-20)

remaining charge fraction = 1.056e-06


In [3]:
# First-order convergence: halving dt should halve the successive charge difference.
end_time=2e-6
coarse=transient_minority_diffusion_fd(length,D,tau,1e20,np.linspace(0,end_time,21),points=181)
medium=transient_minority_diffusion_fd(length,D,tau,1e20,np.linspace(0,end_time,41),points=181)
fine=transient_minority_diffusion_fd(length,D,tau,1e20,np.linspace(0,end_time,81),points=181)
qc, qm, qf = (run["stored_charge_c_m2"][-1] for run in (coarse,medium,fine))
refinement_ratio=abs(qf-qm)/abs(qm-qc)
print(f"successive-difference ratio under dt halving = {refinement_ratio:.3f}")
assert 0.35 < refinement_ratio < 0.65

successive-difference ratio under dt halving = 0.504


In [4]:
from semphy.junctions import charge_storage_figure
from semphy.junction_atlas import small_signal_atlas, switching_atlas

charge_storage_figure()
small_signal_atlas()
switching_atlas()
print("wrote storage validation, small-signal circuit, and switching plate")

wrote storage validation, small-signal circuit, and switching plate
